In [ ]:
# 1. Imports
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [ ]:
# 2. Load the data (change the path if needed)
df = pd.read_csv("adult_income.csv")
print(df.shape)
df.head()

In [ ]:
# 3. Look at the data
df.info()
print(df["income"].value_counts())

In [ ]:
# 4. Simple charts
df["income"].value_counts().plot(kind="bar", title="Income classes")
plt.show()

df.groupby("income")["age"].mean().plot(kind="bar", title="Average age by income")
plt.show()

df.groupby("income")["hours.per.week"].mean().plot(kind="bar", title="Average hours per week by income")
plt.show()

In [ ]:
# 5. Clean the data
df = df.replace("?", "Unknown")                  # missing values are written as "?"
df = df.drop_duplicates()                         # remove duplicate rows
df = df.drop(columns=["fnlwgt", "education"])     # not useful (education.num already has this info)

# target: 1 if income > 50K, else 0
y = (df["income"] == ">50K").astype(int)
X = df.drop(columns="income")

In [ ]:
# 6. Convert text columns to numbers
X = pd.get_dummies(X)
print(X.shape)

In [ ]:
# 7. Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# 8. Train 3 models and compare them
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingClassifier(random_state=42),
}

scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    scores[name] = f1_score(y_test, pred)
    print(name, "| accuracy:", round(accuracy_score(y_test, pred), 4), "| f1:", round(scores[name], 4))

In [ ]:
# 9. Pick the best model (highest F1) and evaluate it
best_name = max(scores, key=scores.get)
best_model = models[best_name]
print("Best model:", best_name)

pred = best_model.predict(X_test)
print(classification_report(y_test, pred, target_names=["<=50K", ">50K"]))
print(confusion_matrix(y_test, pred))

In [ ]:
# 10. Save the model and the column names
joblib.dump(best_model, "income_model.joblib")
joblib.dump(list(X.columns), "model_columns.joblib")
print("Saved!")

In [ ]:
# 11. Load the model and make a prediction
model = joblib.load("income_model.joblib")
columns = joblib.load("model_columns.joblib")

person = {
    "age": 45, "workclass": "Private", "education.num": 14,
    "marital.status": "Married-civ-spouse", "occupation": "Exec-managerial",
    "relationship": "Husband", "race": "White", "sex": "Male",
    "capital.gain": 0, "capital.loss": 0, "hours.per.week": 50,
    "native.country": "United-States",
}

row = pd.get_dummies(pd.DataFrame([person]))
row = row.reindex(columns=columns, fill_value=0)   # match the training columns

result = model.predict(row)[0]
print(">50K" if result == 1 else "<=50K")